# Stage 2: train and evaluate the re-ranker

This notebook trains the re-ranker, a cross-encoder that reads a written address and one candidate G-NAF record together and scores how likely they are the same address. The recipe follows Duarte and Oliveira (2023), *Improving Address Matching Using Siamese Transformer Networks* (EPIA 2023, LNCS 14116; arXiv:2307.02300). The training negatives are the retriever's own wrong answers for synthetic queries. The evaluation is end to end: the re-ranker only sees what the retriever returns for each test query.

- Reads: `output_synthesis/synthetic_addresses.parquet`, `gnaf.duckdb`, the test set `output/misspellings.tsv`, and the retriever's model, index and per-query files in `output_stage1/`.
- Writes: `output_stage2/` (the model, mined negatives, retrieval arrays, result tables and run manifests).
- Run after: 3_train_retriever.ipynb

Cell 7 (mining) and cell 10 (training) are the long cells. `S2_ALLOW_CACHE=1` reuses their saved outputs.

In [ ]:
# Cell 1: install the packages (run once). CrossEncoderTrainer needs
# sentence-transformers 4 or later.
# !pip install -U "sentence-transformers>=4" datasets duckdb pandas rapidfuzz
print("cell 1: run the pip line above once, then skip this cell")

cell 1: run the pip line above once, then skip this cell


## Part 1: mine negatives and train the re-ranker

In [ ]:
# Cell 2: configuration. Every path and setting is here, and most can be
# overridden from the environment. This notebook does not rebuild the index or
# the test set: it reads what 3_train_retriever produced, so both stages use
# the same matchable queries.
# Environment switches:
#   S2_SMOKE=1        mine 200 queries instead of 8,000, a quick check that
#                     the cells run.
#   S2_ALLOW_CACHE=1  cell 10 loads OUT/model instead of training, and cells 6
#                     and 9 skip their timing pilots. Cell 7 reuses its saved
#                     mining files whenever their sidecar (the small JSON file
#                     beside them) matches. The sidecar refuses a different
#                     MINE_N, so S2_SMOKE=1 cannot reuse an 8,000-query run.

import os
from pathlib import Path

def _env(name, default):
    return os.environ.get(name, default)

# Retriever outputs (read only)
S1_OUT     = Path(_env("S1_OUT", "output_stage1"))
S1_MODEL   = S1_OUT / "model"                      # the trained bi-encoder
EMB_NPY    = S1_OUT / "emb_store.npy"              # 15,901,232 x 384 fp32
PID_NPY    = S1_OUT / "emb_pids.npy"               # row i of EMB_NPY is pid i
S1_PERQ    = S1_OUT / "stage1_recall_per_query.tsv"  # matchable test queries, with truth_pid

# Other inputs
DB_PATH          = Path(_env("S2_DB", "gnaf.duckdb"))
SYN_PATH         = next((Path(p) for p in [_env("S2_SYN", ""),
                        "output_synthesis/synthetic_addresses.parquet",
                        "output_synthesis/synthetic_addresses.tsv"]
                        if p and Path(p).exists()), None)
MISSPELLINGS_TSV = next((Path(p) for p in [_env("S2_MISS", ""),
                        "output/misspellings.tsv", "misspellings.tsv"]
                        if p and Path(p).exists()), None)

OUT = Path(_env("S2_OUT", "output_stage2")); OUT.mkdir(exist_ok=True)

# Model and seed
CE_MODEL = _env("CE_MODEL", "cross-encoder/ms-marco-MiniLM-L6-v2")
SEED     = 4640                        # same seed as the synthesiser
SMOKE    = _env("S2_SMOKE", "0") == "1"
ALLOW_CACHE = _env("S2_ALLOW_CACHE", "0") == "1"

# Negative mining: MINE_N synthetic queries go through the trained retriever,
# and the wrong answers it returns become the training negatives.
MINE_N       = 200 if SMOKE else 8_000   # synthetic queries to mine against
MINE_K       = 32                        # candidates retrieved per query
NEG_PER_POS  = 8                         # hardest N kept as negatives
MINE_BATCH   = 250 if SMOKE else 1_000   # queries per index pass; see cell 6
CHUNK_ROWS   = 250_000                   # index rows per block; sets memory use
FILTER_TEST_ANCHORS = True               # see cell 5

# Re-ranker training
EMBED_DIM  = 384
EMBED_BATCH = 64                   # retriever encode batch (queries only)
MAX_LEN    = 64                    # same token limit as the retriever
EPOCHS     = 2
BATCH      = 16
LR         = 2e-5
FP16       = False                 # MPS does not support fp16 training
VAL_FRAC   = 0.10                  # synthetic validation share, split by query
POS_WEIGHT = None                  # None = plain BCE; see cell 10
PREDICT_BATCH = 128

import numpy as np, pandas as pd, re, json, time, zlib
rng = np.random.default_rng(SEED)

for label, p in [("Stage 1 model", S1_MODEL), ("embeddings", EMB_NPY),
                 ("pid array", PID_NPY), ("Stage 1 per-query", S1_PERQ),
                 ("database", DB_PATH)]:
    assert p.exists(), f"{label} not found at {p}; Stage 1 must have been run and locked"
assert SYN_PATH is not None,         "no synthetic file found; set S2_SYN"
assert MISSPELLINGS_TSV is not None, "misspellings.tsv not found; set S2_MISS"

print(f"stage 1 model : {S1_MODEL}")
print(f"index         : {EMB_NPY}")
print(f"synthetic     : {SYN_PATH}")
print(f"test set      : {S1_PERQ} (truth already resolved by Stage 1)")
print(f"smoke mode    : {SMOKE}")
print(f"allow cache   : {ALLOW_CACHE}   (S2_ALLOW_CACHE=1 reuses OUT/model and skips the pilots)")
print(f"mining        : {MINE_N:,} queries x K={MINE_K}, keeping {NEG_PER_POS} negatives each")

stage 1 model : output_stage1/model
index         : output_stage1/emb_store.npy
synthetic     : output_synthesis/synthetic_addresses.parquet
test set      : output_stage1/stage1_recall_per_query.tsv (truth already resolved by Stage 1)
smoke mode    : False
allow cache   : False   (S2_ALLOW_CACHE=1 reuses OUT/model and skips the pilots)
mining        : 8,000 queries x K=32, keeping 8 negatives each


In [ ]:
# Cell 3: open G-NAF read-only and define RENDER, the one place a database
# row becomes an address string. RENDER must match 3_train_retriever byte for
# byte, or the re-ranker would score different strings from the ones the
# retriever indexed. The hash check stops the notebook if it differs.

import duckdb, hashlib
con = duckdb.connect(str(DB_PATH), read_only=True)

RENDER = r"""regexp_replace(trim(concat_ws(' ',
    CASE WHEN g.number_last IS NOT NULL AND g.number_last <> ''
              AND regexp_matches(k.k, '^[0-9]')
         THEN k.k || '-' || g.number_last ELSE k.k END,
    k.street_n, k.type_n, k.suffix_n, k.locality_n, k.state_n, k.postcode)),
    '\s+', ' ', 'g')"""
RJOIN = "LEFT JOIN gnaf_range g ON g.pid = k.pid AND g.number_first = k.k"

_H = hashlib.sha1(RENDER.encode()).hexdigest()
assert _H == "cc573253b3ead7dddebc8393697909d49f27eaa1", (
    f"RENDER has drifted (sha1 {_H}). It must match 3_train_retriever.ipynb "
    "cell 4 byte for byte, or Stage 2 is scoring different strings from the ones "
    "Stage 1 indexed.")

def norm(s):
    """The query normaliser Stage 1 used: upper, commas to spaces, collapse."""
    return re.sub(r"\s+", " ", str(s).upper().replace(",", " ")).strip()

print(f"database ok, RENDER hash verified ({_H[:12]}...)")

database ok, RENDER hash verified (cc573253b3ea...)


In [ ]:
# Cell 4: load the retriever and memory-map its 22.7 GiB index, then define
# batched_topk, which scores many queries against the whole index in one pass.
# The index was built with the model truncating at 64 tokens, so the reloaded
# model is held to the same limit.
# Peak memory is about n_queries * CHUNK_ROWS * 4 bytes for the similarity
# matrix: 1,000 queries x 250,000 rows = 1.0 GB.

import torch
from sentence_transformers import SentenceTransformer

device = "mps" if torch.backends.mps.is_available() else (
         "cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

s1_model = SentenceTransformer(str(S1_MODEL), device=device)
if s1_model.max_seq_length != MAX_LEN:
    print(f"  WARNING: reloaded max_seq_length is {s1_model.max_seq_length}, "
          f"forcing to {MAX_LEN} to match the index build")
    s1_model.max_seq_length = MAX_LEN
print(f"stage 1 model loaded, max_seq_length = {s1_model.max_seq_length}")

# The model's weight-file time, so cells 7 and 12 can refuse saved files made
# with a different retriever.
_w = sorted(S1_MODEL.rglob("*.safetensors")) + sorted(S1_MODEL.rglob("*.bin"))
S1_MTIME = max(p.stat().st_mtime for p in _w) if _w else 0.0

_E = np.load(EMB_NPY, mmap_mode="r"); _P = np.load(PID_NPY)
assert _E.shape[0] == len(_P), f"{_E.shape[0]:,} vectors but {len(_P):,} pids"
assert _E.shape[1] == EMBED_DIM, f"index is {_E.shape[1]}-dim, expected {EMBED_DIM}"
print(f"index         : {_E.shape[0]:,} x {_E.shape[1]} ({_E.dtype})")
if _E.dtype != np.float32:
    print("  WARNING: index is not fp32; the fp16 index cost ~4% of the top-50")
del _E, _P


def batched_topk(queries, k, chunk_rows=CHUNK_ROWS, verbose=True):
    """Top-k index rows for every query, in one pass over the array."""
    E = np.load(EMB_NPY, mmap_mode="r"); P = np.load(PID_NPY); N = E.shape[0]
    Q = s1_model.encode(list(queries), batch_size=EMBED_BATCH, convert_to_numpy=True,
                        normalize_embeddings=True,
                        show_progress_bar=verbose).astype(np.float32)
    nq = len(Q)
    run_sim = np.full((nq, k), -np.inf, dtype=np.float32)
    run_idx = np.zeros((nq, k), dtype=np.int64)
    t0 = time.time()
    for s in range(0, N, chunk_rows):
        block = np.asarray(E[s:s+chunk_rows]).astype(np.float32, copy=False)
        sims = Q @ block.T
        kk = min(k, sims.shape[1])
        part = np.argpartition(-sims, kk-1, axis=1)[:, :kk]
        all_sim = np.concatenate([run_sim, np.take_along_axis(sims, part, axis=1)], axis=1)
        all_idx = np.concatenate([run_idx, part.astype(np.int64) + s], axis=1)
        sel = np.argpartition(-all_sim, k-1, axis=1)[:, :k]
        run_sim = np.take_along_axis(all_sim, sel, axis=1)
        run_idx = np.take_along_axis(all_idx, sel, axis=1)
        if verbose and (s // chunk_rows) % 20 == 0:
            frac = min(s + chunk_rows, N) / N; el = time.time() - t0
            print(f"  {min(s+chunk_rows,N):,}/{N:,}  {el:6.0f}s, "
                  f"~{el/max(frac,1e-9)-el:5.0f}s left")
    order = np.argsort(-run_sim, axis=1)
    run_sim = np.take_along_axis(run_sim, order, axis=1)
    run_idx = np.take_along_axis(run_idx, order, axis=1)
    if verbose:
        el = time.time() - t0
        print(f"  pass done in {el:.0f}s ({el/max(nq,1)*1000:.0f} ms/query amortised)")
    return P[run_idx], run_sim

print("batched_topk ready")

device: mps


stage 1 model loaded, max_seq_length = 64
index         : 15,901,232 x 384 (float32)
batched_topk ready


In [ ]:
# Cell 5: pick MINE_N synthetic positives to mine negatives for, and build the
# set of address strings that answer a test query.
# The retriever's rule is that no G-NAF record answering a test query is used
# as a training target. A mined negative is not a target, so this filter is
# stricter than needed, but it keeps both stages under one rule. The number of
# strings filtered is printed.

syn = (pd.read_parquet(SYN_PATH) if SYN_PATH.suffix == ".parquet"
       else pd.read_csv(SYN_PATH, sep="\t", dtype=str))
syn = syn.dropna(subset=["dirty", "anchor"]).drop_duplicates("dirty")
syn["y"] = (~syn.label.str.startswith("NEG")).astype(int)
print(f"{len(syn):,} synthetic rows | " + syn.label.value_counts().to_dict().__str__())

pos_all = syn[syn.y == 1]
take = min(MINE_N, len(pos_all))
mine = pos_all.sample(take, random_state=SEED)[["dirty", "anchor"]].reset_index(drop=True)

# The synthesiser's own negatives (phantom number, wrong place, other state)
# are added at the same sampling rate as the positives.
neg_all  = syn[syn.y == 0]
frac     = take / max(len(pos_all), 1)
syn_neg  = neg_all.sample(max(1, int(len(neg_all) * frac)),
                          random_state=SEED)[["dirty", "anchor"]].reset_index(drop=True)
print(f"mining pool: {len(mine):,} positives | synthesiser negatives: {len(syn_neg):,}")

# The matchable test queries, taken from the retriever's per-query file.
perq = pd.read_csv(S1_PERQ, sep="\t", dtype={"truth_pid": str})
test_q = perq[["address", "truth_pid", "rank_principal", "top_sim"]].copy()
test_q["q"] = test_q.address.map(norm)
print(f"test set    : {len(test_q):,} matchable queries carried over from Stage 1")

# Render every test truth pid. With the filter on, these strings are never
# used as a negative.
con.register("truth_df", pd.DataFrame({"pid": test_q.truth_pid.unique()}))
_ta = con.sql(f"""
    SELECT t.pid, {RENDER} AS s
    FROM truth_df t JOIN gnaf_key k ON k.pid = t.pid {RJOIN}
    GROUP BY t.pid, s
""").df()
TEST_ANCHORS = set(_ta.s) if FILTER_TEST_ANCHORS else set()
print(f"test-answer guard: {len(TEST_ANCHORS):,} strings "
      f"({'ON' if FILTER_TEST_ANCHORS else 'OFF'})")

100,000 synthetic rows | {'POS': 84971, 'NEG_PHANTOM_NUMBER': 11936, 'NEG_STREET_WRONG_PLACE': 1720, 'NEG_STREET_OTHER_STATE': 1373}


mining pool: 8,000 positives | synthesiser negatives: 1,414
test set    : 789 matchable queries carried over from Stage 1


test-answer guard: 1,139 strings (ON)


In [ ]:
# Cell 6: project the cost of mining. Two small mining passes are timed and
# fitted to cost = a + b * n_queries. One pass over the index has a fixed cost
# (reading 22.7 GiB) plus a per-query cost (the matrix multiply), and one
# timing cannot separate them. About 2 to 4 minutes. Skipped under
# S2_ALLOW_CACHE=1 when the mining files exist.

if ALLOW_CACHE and (OUT / "mine_progress.json").exists():
    print("[cache] S2_ALLOW_CACHE=1 and mine_progress.json exists: mining pilot skipped")
else:
    n_a, n_b = 100, 400
    t0 = time.time(); batched_topk(mine.dirty.head(n_a).tolist(), k=MINE_K, verbose=False); t_a = time.time()-t0
    t0 = time.time(); batched_topk(mine.dirty.head(n_b).tolist(), k=MINE_K, verbose=False); t_b = time.time()-t0

    b = (t_b - t_a) / (n_b - n_a)      # seconds per query (the matrix multiply)
    a = t_a - b * n_a                  # seconds per pass  (the scan)
    passes = int(np.ceil(len(mine) / MINE_BATCH))
    proj = passes * a + len(mine) * b

    print(f"pilot: {n_a} queries {t_a:5.1f}s | {n_b} queries {t_b:5.1f}s")
    print(f"fit  : {a:5.1f}s fixed per index pass + {b*1000:.1f} ms per query")
    print(f"\nPROJECTED FULL MINING RUN")
    print(f"  {len(mine):,} queries in {passes} passes of {MINE_BATCH:,}")
    print(f"  ~{proj/60:.1f} minutes")
    if proj > 3600:
        print("  >1 hour. Lower MINE_N, or raise MINE_BATCH and lower CHUNK_ROWS "
              "to trade memory for fewer passes.")
    print(f"  peak memory this shape: ~{MINE_BATCH*CHUNK_ROWS*4/1e9:.2f} GB for the "
          "similarity matrix")

pilot: 100 queries  59.4s | 400 queries  83.6s
fit  :  51.3s fixed per index pass + 80.7 ms per query

PROJECTED FULL MINING RUN
  8,000 queries in 8 passes of 1,000
  ~17.6 minutes
  peak memory this shape: ~1.00 GB for the similarity matrix


In [ ]:
# Cell 7: mine hard negatives. Every mining query goes through the trained
# retriever and the top MINE_K candidates are kept, so the re-ranker trains on
# the retriever's own mistakes. The saved run took about 21 minutes.
# Results are saved after each batch, so the cell can resume. The sidecar
# records the retriever's weight-file time, and a resume against a different
# retriever stops with an error.

MP, MS = OUT / "mine_pids.npy", OUT / "mine_sims.npy"
SIDE   = OUT / "mine_progress.json"

done = 0
if SIDE.exists() and MP.exists():
    sc = json.loads(SIDE.read_text())
    assert abs(sc["s1_mtime"] - S1_MTIME) < 1.0, (
        "the saved mining run was produced by a DIFFERENT Stage 1 model "
        f"(sidecar {sc['s1_mtime']}, current {S1_MTIME}). Delete "
        f"{MP.name}, {MS.name} and {SIDE.name} and mine again.")
    assert sc["mine_k"] == MINE_K and sc["n_total"] == len(mine), \
        "MINE_K or MINE_N changed since the saved run; delete the files and re-mine"
    done = sc["n_done"]
    print(f"resuming: {done:,} of {len(mine):,} already mined")

if done < len(mine):
    acc_p = [np.load(MP)] if done else []
    acc_s = [np.load(MS)] if done else []
    t0 = time.time()
    for start in range(done, len(mine), MINE_BATCH):
        chunk = mine.dirty.iloc[start:start + MINE_BATCH].tolist()
        p, s = batched_topk(chunk, k=MINE_K, verbose=False)
        acc_p.append(p); acc_s.append(s)
        np.save(MP, np.concatenate(acc_p)); np.save(MS, np.concatenate(acc_s))
        n_done = start + len(chunk)
        SIDE.write_text(json.dumps(dict(n_done=n_done, n_total=len(mine),
                                        mine_k=MINE_K, s1_mtime=S1_MTIME)))
        el = time.time() - t0
        print(f"  {n_done:,}/{len(mine):,}  {el/60:5.1f} min elapsed, "
              f"~{el/(n_done-done)*(len(mine)-n_done)/60:5.1f} min left")

mined_pids, mined_sims = np.load(MP), np.load(MS)
assert mined_pids.shape == (len(mine), MINE_K), \
    f"mined array is {mined_pids.shape}, expected {(len(mine), MINE_K)}"
print(f"\nmined: {mined_pids.shape[0]:,} queries x {mined_pids.shape[1]} candidates")

# How similar the retriever's best answer is for the mining queries. If the
# synthetic queries are much easier than the real ones, the mined negatives
# are easier too.
_first = pd.Series(mined_sims[:, 0])
print(f"top-1 similarity on mining queries: median {_first.median():.3f}, "
      f"p10 {_first.quantile(0.10):.3f}, p90 {_first.quantile(0.90):.3f}")

  1,000/8,000    2.6 min elapsed, ~ 18.2 min left


  2,000/8,000    5.1 min elapsed, ~ 15.4 min left


  3,000/8,000    7.7 min elapsed, ~ 12.8 min left


  4,000/8,000   10.3 min elapsed, ~ 10.3 min left


  5,000/8,000   12.9 min elapsed, ~  7.7 min left


  6,000/8,000   15.4 min elapsed, ~  5.1 min left


  7,000/8,000   18.0 min elapsed, ~  2.6 min left


  8,000/8,000   20.6 min elapsed, ~  0.0 min left

mined: 8,000 queries x 32 candidates
top-1 similarity on mining queries: median 0.869, p10 0.716, p90 0.969


In [ ]:
# Cell 8: assemble the training pairs (query, candidate, label). The positive
# is the mutated address with its true record. The negatives are the
# highest-scoring records the retriever returned that are not the true record.
# Negatives are filtered by rendered string, not by pid. RENDER drops unit and
# level, so every unit in a building renders to the same string as the truth.
# Filtering by pid would label those strings wrong when they are right.

from collections import defaultdict

need = sorted({b.decode() if isinstance(b, bytes) else str(b)
               for row in mined_pids for b in row})
print(f"rendering {len(need):,} distinct retrieved pids")
con.register("need_df", pd.DataFrame({"pid": need}))
_r = con.sql(f"""
    SELECT n.pid, min({RENDER}) AS s
    FROM need_df n JOIN gnaf_key k ON k.pid = n.pid {RJOIN}
    GROUP BY n.pid
""").df()
REND = dict(zip(_r.pid, _r.s))
print(f"  rendered {len(REND):,} ({len(need)-len(REND):,} pids had no key row)")

rows, stat = [], defaultdict(int)
for i, r in enumerate(mine.itertuples()):
    rows.append((r.dirty, r.anchor, 1))
    seen, kept = {r.anchor}, 0
    for pid_b, sim in zip(mined_pids[i], mined_sims[i]):
        if kept >= NEG_PER_POS:
            break
        s = REND.get(pid_b.decode() if isinstance(pid_b, bytes) else str(pid_b))
        if s is None:          stat["unrendered"] += 1;   continue
        if s in seen:          stat["same_as_truth_or_dup"] += 1; continue
        if s in TEST_ANCHORS:  stat["test_answer_skipped"] += 1;  continue
        seen.add(s); kept += 1
        rows.append((r.dirty, s, 0))
    stat["negatives_kept"] += kept
    if kept < NEG_PER_POS:
        stat["queries_short_of_quota"] += 1

for r in syn_neg.itertuples():
    rows.append((r.dirty, r.anchor, 0)); stat["synthesiser_negatives"] += 1

pairs = (pd.DataFrame(rows, columns=["query", "candidate", "label"])
           .drop_duplicates(["query", "candidate"]).reset_index(drop=True))

print("\nassembly:")
for k, v in sorted(stat.items()):
    print(f"  {k:26s} {v:,}")
n1, n0 = int((pairs.label == 1).sum()), int((pairs.label == 0).sum())
print(f"\npairs: {len(pairs):,}  ({n1:,} positive, {n0:,} negative, 1:{n0/max(n1,1):.1f})")
assert n1 > 0 and n0 > 0, "one class is empty; check the mining output"

# Split by query string, so no mutated address appears on both sides.
pairs["grp"] = pairs["query"].map(lambda s: zlib.crc32(s.encode()) % 100)
train_df = pairs[pairs.grp >= VAL_FRAC * 100].drop(columns="grp").reset_index(drop=True)
val_df   = pairs[pairs.grp <  VAL_FRAC * 100].drop(columns="grp").reset_index(drop=True)
assert not (set(train_df["query"]) & set(val_df["query"])), "query leaked across the split"
print(f"train {len(train_df):,} / val {len(val_df):,} (disjoint by query string)")
print("\nsample rows:")
print(pairs.sample(6, random_state=SEED).to_string(index=False))

rendering 247,258 distinct retrieved pids


  rendered 247,258 (0 pids had no key row)



assembly:
  negatives_kept             51,615
  queries_short_of_quota     1,961
  same_as_truth_or_dup       81,593
  synthesiser_negatives      1,414
  test_answer_skipped        51

pairs: 61,029  (8,000 positive, 53,029 negative, 1:6.6)
train 54,679 / val 6,350 (disjoint by query string)

sample rows:
                                          query                                     candidate  label  grp
SHOP 24 87 HELER STREET BRUNSWICK WEST VIC 3055      87 HELLER STREET BRUNSWICK WEST VIC 3055      1   62
              4/56 RDLANDS ROAD COROWA NSW 2646          56 CORVUS ROAD HINCHINBROOK NSW 2168      0    6
 287/289 SPRINGVALE ROAD SOUTH DONVALE VIC 3111       289 SPRINGVALE ROAD NUNAWADING VIC 3131      0   88
       LOT 3 IODIDE STREET BROKEN HILI NSW 2880      LOT 3 IODIDE STREET BROKEN HILL NSW 2880      1   13
       2426 KEIGHERY DR ALTONA MEADOWS VIC 3978 2420 FRANKSTON-FLINDERS ROAD BITTERN VIC 3918      0   71
      52 BULWARA ROAD POINT PIPER EAST NSW 2009         

In [ ]:
# Cell 9: project the cost of training. Runs 50 optimiser steps on the
# training pairs, times them, and projects the full run. About 1 minute.
# Under S2_ALLOW_CACHE=1 with a saved model the pilot is skipped, but to_ds()
# is still defined.

from datasets import Dataset
from sentence_transformers.cross_encoder import (
    CrossEncoder, CrossEncoderTrainer, CrossEncoderTrainingArguments)
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss

def to_ds(df):
    return Dataset.from_pandas(
        df.assign(label=df.label.astype("float32"))[["query", "candidate", "label"]],
        preserve_index=False)

if ALLOW_CACHE and (OUT / "model").exists():
    print("[cache] S2_ALLOW_CACHE=1 and OUT/model exists: training pilot skipped")
else:
    _probe = CrossEncoder(CE_MODEL, num_labels=1, max_length=MAX_LEN, device=device)
    _args = CrossEncoderTrainingArguments(
        output_dir=str(OUT / "_probe"), max_steps=50,
        per_device_train_batch_size=BATCH, learning_rate=LR,
        fp16=FP16, bf16=False, eval_strategy="no", save_strategy="no",
        logging_steps=50, seed=SEED, dataloader_num_workers=0, report_to="none")
    _t0 = time.time()
    CrossEncoderTrainer(model=_probe, args=_args,
                        train_dataset=to_ds(train_df.head(50 * BATCH)),
                        loss=BinaryCrossEntropyLoss(_probe)).train()
    _sec = (time.time() - _t0) / 50

    steps = int(np.ceil(len(train_df) / BATCH)) * EPOCHS
    print(f"\nmeasured  : {_sec:.2f} s per step at batch {BATCH}, max_len {MAX_LEN}")
    print(f"full run  : {steps:,} steps over {EPOCHS} epochs")
    print(f"PROJECTED : {steps*_sec/3600:.2f} hours ({steps*_sec/60:.0f} min)")
    del _probe

Step,Training Loss
50,0.516832



measured  : 0.09 s per step at batch 16, max_len 64
full run  : 6,836 steps over 2 epochs
PROJECTED : 0.18 hours (11 min)


In [ ]:
# Cell 10: fine-tune the re-ranker. The model reads "[CLS] query [SEP]
# candidate [SEP]" and a one-unit head learns the 0/1 label under binary
# cross-entropy (BCE). The saved run trained in about 10 minutes. A checkpoint
# is saved each epoch, and a manifest is written beside the weights.
# POS_WEIGHT: the pairs are imbalanced towards negatives, so plain BCE can
# score low everywhere. Re-ranking needs only the order, and tau is fitted
# separately (cell 17), so plain BCE is used.
# Under S2_ALLOW_CACHE=1 the saved model is loaded and the manifest is kept.

import transformers
if ALLOW_CACHE and (OUT / "model").exists():
    ce_model = CrossEncoder(str(OUT / "model"), num_labels=1, max_length=MAX_LEN, device=device)
    print(f"[cache] loaded existing cross-encoder from {OUT/'model'} (S2_ALLOW_CACHE=1); "
          "s2_manifest.json left untouched")
else:
    ce_model = CrossEncoder(CE_MODEL, num_labels=1, max_length=MAX_LEN, device=device)

    _tf5  = int(transformers.__version__.split(".")[0]) >= 5
    _warm = dict(warmup_steps=0.1) if _tf5 else dict(warmup_ratio=0.1)

    loss = (BinaryCrossEntropyLoss(ce_model, pos_weight=torch.tensor(float(POS_WEIGHT)))
            if POS_WEIGHT else BinaryCrossEntropyLoss(ce_model))

    args = CrossEncoderTrainingArguments(
        output_dir=str(OUT / "checkpoints"),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH * 4,
        learning_rate=LR, **_warm,
        fp16=FP16, bf16=False,
        eval_strategy="epoch",
        save_strategy="epoch", save_total_limit=2,
        logging_steps=50, seed=SEED,
        dataloader_num_workers=0,
        report_to="none")

    trainer = CrossEncoderTrainer(model=ce_model, args=args,
                                  train_dataset=to_ds(train_df),
                                  eval_dataset=to_ds(val_df), loss=loss)
    _t0 = time.time()
    trainer.train()
    _mins = (time.time() - _t0) / 60

    ce_model.save_pretrained(str(OUT / "model"))
    (OUT / "s2_manifest.json").write_text(json.dumps(dict(
        base_model=CE_MODEL, epochs=EPOCHS, batch=BATCH, lr=LR, max_len=MAX_LEN,
        pos_weight=POS_WEIGHT, n_train_pairs=len(train_df), n_val_pairs=len(val_df),
        mine_n=len(mine), mine_k=MINE_K, neg_per_pos=NEG_PER_POS,
        filter_test_anchors=FILTER_TEST_ANCHORS, s1_mtime=S1_MTIME,
        train_minutes=round(_mins, 1),
        timestamp=time.strftime("%Y-%m-%d %H:%M:%S")), indent=2))
    print(f"\ntrained in {_mins:.0f} min, saved to {OUT/'model'}")

Epoch,Training Loss,Validation Loss
1,0.172015,0.126018
2,0.157495,0.129111



trained in 10 min, saved to output_stage2/model


## Part 2: end-to-end evaluation on the test set

In [ ]:
# Cell 11: load the trained re-ranker and put every test query, matchable or
# not, into one frame (allq). The kinds are:
#   matchable  a G-NAF record answers it (from the retriever's per-query file,
#              so both stages use the same denominator)
#   no_number  unmatchable: the street number does not exist on that street
#   no_record  unmatchable: the harvest could not match it at all
#   excluded   rows whose adjudication column says 'excluded'; kept so the
#              row count equals the file, but scored nowhere

from sentence_transformers.cross_encoder import CrossEncoder
ce = CrossEncoder(str(OUT / "model"), max_length=MAX_LEN, device=device)
print(f"cross-encoder loaded from {OUT/'model'}")

full = pd.read_csv(MISSPELLINGS_TSV, sep="\t", dtype=str).fillna("")
if "adjudication" in full.columns:
    _exc = full.adjudication.str.contains("excluded", regex=False)
else:
    _exc = pd.Series(False, index=full.index)
n_excluded = int(_exc.sum())
print(f"{n_excluded} rows excluded by adjudication (kept in allq as population 'excluded', scored nowhere):")
for _a in full.address[_exc]:
    print(f"    {_a}")
excl = full[_exc].copy()
excl["kind"] = "excluded"; excl["population"] = "excluded"; excl["truth_pid"] = ""

no_rec = full.reason.str.contains("Cannot be matched") & ~_exc
no_num = full.reason.str.contains("Street number does not exist") & ~_exc

unmatch = full[no_rec | no_num].copy()
unmatch["kind"] = np.where(unmatch.reason.str.contains("Cannot be matched"),
                           "no_record", "no_number")
unmatch["population"] = "unmatchable"
unmatch["truth_pid"] = ""

match = test_q.copy()                       # from cell 5
match["kind"] = "matchable"
match["population"] = "matchable"
match["reason"] = ""

allq = pd.concat([
    match[["address", "q", "truth_pid", "population", "kind", "reason"]],
    unmatch.assign(q=unmatch.address.map(norm))
           [["address", "q", "truth_pid", "population", "kind", "reason"]],
    excl.assign(q=excl.address.map(norm))
           [["address", "q", "truth_pid", "population", "kind", "reason"]],
], ignore_index=True)
allq["uid"] = allq.index

print(f"\n{'population':14s} {'n':>5s}")
for k, g in allq.groupby("kind"):
    print(f"{k:14s} {len(g):5d}")
print(f"{'TOTAL':14s} {len(allq):5d}")
assert len(allq) == len(full), (
    f"{len(allq)} queries assembled but misspellings.tsv has {len(full)}; "
    "a row was lost or double-counted. If the matchable count is smaller than "
    "the file's, the Stage 1 per-query file predates this test set: re-run "
    "Stage 1 Part 2 first (or point S2_MISS at the test set Stage 1 used).")

# The retriever alone, as the reference for everything below.
s1_top1 = float((match.rank_principal == 1).mean())
print(f"\nStage 1 alone, top-1        : {s1_top1:.4f}")
for kk in [10, 20, 50, 100]:
    r = float(((match.rank_principal > 0) & (match.rank_principal <= kk)).mean())
    print(f"Stage 1 recall@{kk:<4d}        : {r:.4f}   <- ceiling for Stage 2 at K={kk}")

cross-encoder loaded from output_stage2/model
0 rows excluded by adjudication (kept in allq as population 'excluded', scored nowhere):

population         n
matchable        789
no_number        129
no_record         32
TOTAL            950

Stage 1 alone, top-1        : 0.5463
Stage 1 recall@10          : 0.7541   <- ceiling for Stage 2 at K=10
Stage 1 recall@20          : 0.8213   <- ceiling for Stage 2 at K=20
Stage 1 recall@50          : 0.9125   <- ceiling for Stage 2 at K=50
Stage 1 recall@100         : 0.9455   <- ceiling for Stage 2 at K=100


In [ ]:
# Cell 12: run the retriever over every test query (950 in the saved run),
# 500 deep. This makes the evaluation end to end: if the retriever never
# returns the true record, that query counts as wrong.
# Only the top 100 are scored, but several pids often share one rendered
# string, so 100 distinct addresses need more than 100 retrieved pids. The
# index is read once whatever K is. One pass took about 2.5 minutes.
# Saved results are reused only if the sidecar matches the retriever, the
# number of queries, K and a sha1 of the query list in order.

import hashlib
RETRIEVE_K = 500
EP, ES = OUT / "eval_pids.npy", OUT / "eval_sims.npy"
ESIDE  = OUT / "eval_retrieval.json"
Q_HASH = hashlib.sha1("\n".join(allq.q.tolist()).encode()).hexdigest()

_fresh, _why = True, "no cached retrieval"
if ESIDE.exists() and EP.exists():
    sc = json.loads(ESIDE.read_text())
    _checks = {"model mtime": abs(sc.get("s1_mtime", -1) - S1_MTIME) < 1.0,
               "n queries":   sc.get("n") == len(allq),
               "K":           sc.get("k") == RETRIEVE_K,
               "query list":  sc.get("q_hash") == Q_HASH}
    _fresh = not all(_checks.values())
    _why   = ", ".join(k for k, ok in _checks.items() if not ok) or "all checks pass"
    print("cached retrieval is " + (f"stale ({_why}), re-running" if _fresh
                                    else f"current ({_why}), reusing"))
else:
    print(f"{_why}: running")

if _fresh:
    _t0 = time.time()
    ev_pids, ev_sims = batched_topk(allq.q.tolist(), k=RETRIEVE_K)
    np.save(EP, ev_pids); np.save(ES, ev_sims)
    ESIDE.write_text(json.dumps(dict(s1_mtime=S1_MTIME, n=len(allq),
                                     k=RETRIEVE_K, q_hash=Q_HASH,
                                     minutes=round((time.time()-_t0)/60, 1))))
ev_pids, ev_sims = np.load(EP), np.load(ES)
assert ev_pids.shape == (len(allq), RETRIEVE_K)
print(f"retrieved: {ev_pids.shape[0]:,} queries x {ev_pids.shape[1]} candidates")

no cached retrieval: running


  250,000/15,901,232       3s, ~  176s left


  5,250,000/15,901,232      48s, ~   97s left


  10,250,000/15,901,232      96s, ~   53s left


  15,250,000/15,901,232     145s, ~    6s left


  pass done in 151s (159 ms/query amortised)
retrieved: 950 queries x 500 candidates


In [ ]:
# Cell 13: build the two candidate sets the re-ranker is given, both capped at
# K_MAX so a smaller K is read off by truncation:
#   raw    the top K_MAX pids in the retriever's order
#   dedup  the top K_MAX distinct rendered strings, each kept with its
#          best-ranked pid
# Several pids often render to the same string (cell 13b prints how many), so
# a raw top K holds fewer distinct addresses than K.
# A pid can carry both a "LOT 619 ..." key and a street-number key. The
# retriever indexed the street-number form, so the SQL prefers it explicitly.

K_MAX = 100

need = sorted({b.decode() if isinstance(b, bytes) else str(b)
               for row in ev_pids for b in row} | set(match.truth_pid))
print(f"resolving {len(need):,} distinct pids")
con.register("need_df", pd.DataFrame({"pid": need}))
_res = con.sql(f"""
    SELECT n.pid,
           coalesce(min(CASE WHEN k.k NOT LIKE 'LOT %' THEN {RENDER} END),
                    min({RENDER}))                                   AS s,
           min(coalesce(nullif(CAST(k.PRINCIPAL_PID AS VARCHAR), ''), n.pid)) AS principal
    FROM need_df n JOIN gnaf_key k ON k.pid = n.pid {RJOIN}
    GROUP BY n.pid
""").df()
REND2 = dict(zip(_res.pid, _res.s))
P2P   = dict(zip(_res.pid, _res.principal))
print(f"  rendered {len(REND2):,}, principal map {len(P2P):,}")

_dec = np.vectorize(lambda b: b.decode() if isinstance(b, bytes) else str(b))
raw_rows, ded_rows, ded_short = [], [], 0
for i in range(len(allq)):
    pl = _dec(ev_pids[i]); sl = ev_sims[i]
    for r in range(min(K_MAX, len(pl))):
        raw_rows.append((i, r + 1, pl[r], REND2.get(pl[r], ""), float(sl[r])))
    seen = set()
    for r in range(len(pl)):
        s = REND2.get(pl[r], "")
        if not s or s in seen:
            continue
        seen.add(s)
        ded_rows.append((i, len(seen), pl[r], s, float(sl[r])))
        if len(seen) >= K_MAX:
            break
    if len(seen) < K_MAX:
        ded_short += 1

cols = ["uid", "rank", "pid", "cand", "sim"]
cand_raw   = pd.DataFrame(raw_rows, columns=cols)
cand_dedup = pd.DataFrame(ded_rows, columns=cols)

print(f"\nraw   : {len(cand_raw):,} pairs ({len(cand_raw)/len(allq):.1f} per query)")
print(f"dedup : {len(cand_dedup):,} pairs ({len(cand_dedup)/len(allq):.1f} per query)")
print(f"        {ded_short:,} queries ({ded_short/len(allq):.1%}) could not reach "
      f"{K_MAX} distinct addresses within {RETRIEVE_K} retrieved pids")
_d50 = cand_dedup[cand_dedup["rank"] <= 50].groupby("uid")["rank"].max()
print(f"distinct addresses inside raw K=50 : median "
      f"{cand_raw[cand_raw['rank']<=50].groupby('uid').cand.nunique().median():.0f} of 50")

resolving 435,875 distinct pids


  rendered 435,875, principal map 435,875

raw   : 95,000 pairs (100.0 per query)
dedup : 84,686 pairs (89.1 per query)
        200 queries (21.1%) could not reach 100 distinct addresses within 500 retrieved pids
distinct addresses inside raw K=50 : median 25 of 50


In [ ]:
# Cell 13b: how many distinct addresses a raw top K holds (print only).
print("distinct addresses inside a raw top-K:")
for K in [10, 20, 50, 100]:
    d = cand_raw[cand_raw["rank"] <= K].groupby("uid").cand.nunique()
    print(f"  K={K:<4d} median {d.median():5.0f}  mean {d.mean():5.1f}  "
          f"collapse {K/max(d.mean(),1e-9):.2f} pids per distinct address")

distinct addresses inside a raw top-K:
  K=10   median     7  mean   6.0  collapse 1.67 pids per distinct address
  K=20   median    12  mean  10.9  collapse 1.83 pids per distinct address
  K=50   median    25  mean  25.2  collapse 1.98 pids per distinct address
  K=100  median    51  mean  49.0  collapse 2.04 pids per distinct address


In [ ]:
# Cell 14: score both candidate sets with the re-ranker and with a fuzzy
# baseline (RapidFuzz WRatio) on the same pairs, so any difference comes from
# the scorer. CrossEncoder.predict returns raw logits. The threshold in cell 17
# needs a 0 to 1 score, so the sigmoid is applied here explicitly.
# About 180,000 pairs, 1.5 minutes in the saved run.

from rapidfuzz import fuzz
QMAP = dict(zip(allq.uid, allq.q))

def score(df, name):
    qs = df.uid.map(QMAP).tolist(); cs = df.cand.tolist()
    t0 = time.time()
    logits = ce.predict(list(zip(qs, cs)), batch_size=PREDICT_BATCH,
                        show_progress_bar=True)
    df["s_model"] = 1.0 / (1.0 + np.exp(-np.asarray(logits, dtype=np.float64)))
    df["s_fuzz"]  = [fuzz.WRatio(a, b) / 100.0 for a, b in zip(qs, cs)]
    print(f"{name}: {len(df):,} pairs in {(time.time()-t0)/60:.1f} min")
    return df

cand_raw   = score(cand_raw,   "raw  ")
cand_dedup = score(cand_dedup, "dedup")
print(f"\nmodel score distribution (raw set): "
      f"p5 {cand_raw.s_model.quantile(.05):.3f}  "
      f"p50 {cand_raw.s_model.median():.3f}  "
      f"p95 {cand_raw.s_model.quantile(.95):.3f}")

raw  : 95,000 pairs in 0.8 min


dedup: 84,686 pairs in 0.7 min

model score distribution (raw set): p5 0.000  p50 0.001  p95 0.967


In [ ]:
# Cell 15: the end-to-end table. For each K, each scorer picks its top
# candidate from the retriever's top K, and the pick is checked against the
# true record. The denominator is every matchable query, so a query whose true
# record was never retrieved counts as wrong. Two correctness rules:
#   pid     the chosen pid is the true record (alias pids mapped to their
#           principal)
#   string  the chosen record renders to the same string as the truth. RENDER
#           drops unit and level, so the gap between the two rules is the cost
#           of that. Only this rule applies to the deduplicated sets.
# Ties in score fall back to the retriever's rank (stable sort).

TRUTH_PID = dict(zip(match.address, match.truth_pid))
uid2addr  = dict(zip(allq.uid, allq.address))
m_uids    = allq.loc[allq.population == "matchable", "uid"].to_numpy()

truth_pr  = {u: P2P.get(TRUTH_PID[uid2addr[u]], TRUTH_PID[uid2addr[u]]) for u in m_uids}
truth_str = {u: REND2.get(TRUTH_PID[uid2addr[u]], "") for u in m_uids}

def wilson(p, n, z=1.96):
    d = 1 + z*z/n; c = p + z*z/(2*n); h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return (c - h)/d, (c + h)/d

def evaluate(df, K, col, variant):
    sub = df[(df["rank"] <= K) & (df.uid.isin(m_uids))]
    ordered = sub.sort_values(["uid", col, "rank"],
                              ascending=[True, False, True], kind="mergesort")
    top = ordered.groupby("uid").head(1).set_index("uid")
    hit_pid = hit_str = 0
    rr, r5, r10 = [], 0, 0
    for u in m_uids:
        if u not in top.index:
            rr.append(0.0); continue
        row = top.loc[u]
        if P2P.get(row.pid, row.pid) == truth_pr[u]: hit_pid += 1
        if row.cand == truth_str[u]:                hit_str += 1
        g = ordered[ordered.uid == u]
        pos = np.flatnonzero(g.cand.to_numpy() == truth_str[u])
        if len(pos):
            rr.append(1.0 / (pos[0] + 1)); r5 += pos[0] < 5; r10 += pos[0] < 10
        else:
            rr.append(0.0)
    n = len(m_uids)
    lo, hi = wilson(hit_str / n, n)
    return dict(variant=variant, scorer=col.replace("s_", ""), K=K,
                top1_pid=hit_pid / n if variant == "raw" else np.nan,
                top1_str=hit_str / n, ci_lo=lo, ci_hi=hi,
                mrr=float(np.mean(rr)), r_at_5=r5 / n, r_at_10=r10 / n)

rows = []
for K in [10, 20, 50, 100]:
    for col in ["s_model", "s_fuzz"]:
        rows.append(evaluate(cand_raw,   K, col, "raw"))
        rows.append(evaluate(cand_dedup, K, col, "dedup"))
e2e = pd.DataFrame(rows)

print(f"END-TO-END, all {len(m_uids)} matchable queries in the denominator")
print(f"Stage 1 alone top-1: {s1_top1:.4f}\n")
print(e2e.round(4).to_string(index=False))
e2e.to_csv(OUT / "stage2_end_to_end.tsv", sep="\t", index=False)

best = e2e[(e2e.scorer == "model") & (e2e.variant == "raw")].nlargest(1, "top1_str").iloc[0]
print(f"\n*** best raw configuration: K={int(best.K)}, top-1 (string) "
      f"{best.top1_str:.4f} [{best.ci_lo:.3f}, {best.ci_hi:.3f}], "
      f"top-1 (pid) {best.top1_pid:.4f} ***")
print(f"    the pid-vs-string gap of {best.top1_str-best.top1_pid:.4f} is the "
      "render collapse, measured end to end")

END-TO-END, all 789 matchable queries in the denominator
Stage 1 alone top-1: 0.5463

variant scorer   K  top1_pid  top1_str  ci_lo  ci_hi    mrr  r_at_5  r_at_10
    raw  model  10    0.5323    0.8276 0.7997 0.8524 0.8562  0.8872   0.9075
  dedup  model  10       NaN    0.8162 0.7877 0.8417 0.8673  0.9278   0.9392
    raw   fuzz  10    0.4728    0.7490 0.7176 0.7780 0.7980  0.8657   0.9075
  dedup   fuzz  10       NaN    0.6781 0.6447 0.7097 0.7568  0.8720   0.9392
    raw  model  20    0.5285    0.8188 0.7904 0.8441 0.8519  0.8897   0.9049
  dedup  model  20       NaN    0.8099 0.7810 0.8357 0.8625  0.9240   0.9430
    raw   fuzz  20    0.4588    0.7262 0.6941 0.7562 0.7790  0.8327   0.8796
  dedup   fuzz  20       NaN    0.6515 0.6175 0.6839 0.7301  0.8302   0.9075
    raw  model  50    0.5222    0.8137 0.7850 0.8393 0.8485  0.8910   0.9024
  dedup  model  50       NaN    0.7997 0.7704 0.8262 0.8520  0.9163   0.9316
    raw   fuzz  50    0.4360    0.6717 0.6382 0.7036 0.7294  0.7921

In [ ]:
# Cell 15b: the retriever alone on the same string rule as cell 15, then the
# retriever against retriever plus re-ranker on paired per-query outcomes
# (McNemar's exact test). If cand_raw has no scores (cell 13 re-run after
# cell 14), the re-ranker's picks are read from stage2_predictions.tsv, which
# cell 17 writes.

import math

for _n in ["cand_raw", "truth_str", "m_uids", "uid2addr", "s1_top1", "match"]:
    assert _n in dir(), (f"{_n} is not in memory. Re-run cells 11 to 15 "
                         "(cell 14 can be skipped if stage2_predictions.tsv exists).")
assert {"uid", "rank", "cand"} <= set(cand_raw.columns), \
    "cand_raw is missing its base columns; re-run cell 13"

# The retriever's own order, scored on the string rule
gg = (cand_raw[cand_raw.uid.isin(m_uids)]
      .sort_values(["uid", "rank"], kind="mergesort"))
s1_str_rank = {}
for uid, grp in gg.groupby("uid", sort=False):
    pos = np.flatnonzero(grp.cand.to_numpy() == truth_str[uid])
    s1_str_rank[uid] = int(grp["rank"].to_numpy()[pos[0]]) if len(pos) else -1

r = np.array([s1_str_rank[u] for u in m_uids])
n = len(m_uids)
s1_str_top1 = float((r == 1).mean())

print("STAGE 1 ALONE, THE TWO RULERS SIDE BY SIDE")
print(f"  {'':22s} {'pid rule':>10s} {'string rule':>12s}")
print(f"  {'top-1':22s} {s1_top1:10.4f} {s1_str_top1:12.4f}")
for K in [10, 20, 50, 100]:
    pid_r = float(((match.rank_principal > 0) & (match.rank_principal <= K)).mean())
    str_r = float(((r > 0) & (r <= K)).mean())
    print(f"  {'recall@'+str(K):22s} {pid_r:10.4f} {str_r:12.4f}")
print(f"\n  the string rule is easier by {s1_str_top1 - s1_top1:.4f} at top-1 "
      f"({round((s1_str_top1-s1_top1)*n)} queries).")
print("  that difference is the render collapse, and it belongs to Stage 1.")

# The re-ranker's picks, from memory if scored, from disk if not
K_REPORT = int(best.K)   # the K cell 15 chose
if "s_model" in cand_raw.columns:
    sub = cand_raw[(cand_raw["rank"] <= K_REPORT) & (cand_raw.uid.isin(m_uids))]
    top = (sub.sort_values(["uid", "s_model", "rank"],
                           ascending=[True, False, True], kind="mergesort")
              .groupby("uid").head(1))
    pick = dict(zip(top.uid, top.cand))
    print(f"\n(Stage 2 picks taken from the scored cand_raw in memory)")
else:
    PRED = OUT / "stage2_predictions.tsv"
    assert PRED.exists(), (
        "cand_raw has no scores and stage2_predictions.tsv is missing. "
        "Re-run cell 14 then cell 17.")
    pr = pd.read_csv(PRED, sep="\t")
    pick = dict(zip(pr.uid, pr.cand.astype(str)))
    print(f"\n(cand_raw was rebuilt unscored; Stage 2 picks read from {PRED.name})")

s2_hit = np.array([pick.get(u, "") == truth_str[u] for u in m_uids])
s1_hit = (r == 1)

print(f"\nSTRING RULE, LIKE FOR LIKE, K={K_REPORT}")
print(f"  Stage 1 alone      : {s1_hit.mean():.4f}  ({int(s1_hit.sum()):3d} of {n})")
print(f"  Stage 1 + Stage 2  : {s2_hit.mean():.4f}  ({int(s2_hit.sum()):3d} of {n})")
print(f"  difference         : {s2_hit.mean()-s1_hit.mean():+.4f}  "
      f"({int(s2_hit.sum()-s1_hit.sum()):+d} queries)")

# McNemar's exact test on the paired outcomes
b = int((s1_hit & ~s2_hit).sum())     # retriever right, re-ranker lost it
c = int((~s1_hit & s2_hit).sum())     # re-ranker rescued it
def mcnemar_exact(b, c):
    m = b + c
    if m == 0: return 1.0
    k = min(b, c)
    return min(1.0, 2 * sum(math.comb(m, i) for i in range(k + 1)) / (2 ** m))
p = mcnemar_exact(b, c)
print(f"\nMcNEMAR (paired, exact binomial)")
print(f"  both right {int((s1_hit & s2_hit).sum()):3d} | "
      f"both wrong {int((~s1_hit & ~s2_hit).sum()):3d}")
print(f"  Stage 2 lost {b:3d} | Stage 2 rescued {c:3d} | discordant {b+c}")
print(f"  two-sided p = {p:.6f}" +
      ("   SIGNIFICANT at 0.05" if p < 0.05 else "   not significant at 0.05"))

STAGE 1 ALONE, THE TWO RULERS SIDE BY SIDE
                           pid rule  string rule
  top-1                      0.5463       0.8124
  recall@10                  0.7541       0.9075
  recall@20                  0.8213       0.9252
  recall@50                  0.9125       0.9442
  recall@100                 0.9455       0.9544

  the string rule is easier by 0.2662 at top-1 (210 queries).
  that difference is the render collapse, and it belongs to Stage 1.

(Stage 2 picks taken from the scored cand_raw in memory)

STRING RULE, LIKE FOR LIKE, K=10
  Stage 1 alone      : 0.8124  (641 of 789)
  Stage 1 + Stage 2  : 0.8276  (653 of 789)
  difference         : +0.0152  (+12 queries)

McNEMAR (paired, exact binomial)
  both right 610 | both wrong 105
  Stage 2 lost  31 | Stage 2 rescued  43 | discordant 74
  two-sided p = 0.200678   not significant at 0.05


In [ ]:
# Cell 15d: would putting units back into RENDER help? For each matchable
# query it crosses two facts:
#   (a) does the written address name a unit, level, flat, suite or shop?
#   (b) does the true record share its rendered string with other records?
# Unit written and string shared: only these could a RENDER change rescue.
# No unit written and string shared: a RENDER change makes these harder,
# because the index would then need a unit the query does not give.
# Unique string: not affected either way.

UNIT_RE = re.compile(
    r"(^|\s)(UNIT|FLAT|APT|APARTMENT|SUITE|LEVEL|LVL|FLOOR|SHOP|OFFICE|ROOM)\s*\d"
    r"|^\s*[A-Z]?\d+\s*/\s*\d"            # "1/8", "G01/1"
    r"|(^|\s)[A-Z]?\d+\s*/\s*\d", re.I)

# How many pids share each true record's rendered string
con.register("ts_df", pd.DataFrame({"s": sorted(set(truth_str.values()))}))
grp = con.sql(f"""
    SELECT t.s, count(*) AS n_pids
    FROM ts_df t
    JOIN (SELECT k.pid, coalesce(min(CASE WHEN k.k NOT LIKE 'LOT %' THEN {RENDER} END),
                                 min({RENDER})) AS s
          FROM gnaf_key k {RJOIN} GROUP BY k.pid) r ON r.s = t.s
    GROUP BY t.s
""").df()
GSZ = dict(zip(grp.s, grp.n_pids))

rows = []
for u in m_uids:
    a = uid2addr[u]
    rows.append(dict(address=a,
                     writes_unit=bool(UNIT_RE.search(a)),
                     group_size=int(GSZ.get(truth_str[u], 1))))
dec = pd.DataFrame(rows)
dec["shared"] = dec.group_size > 1

n = len(dec)
print(f"OF {n} MATCHABLE QUERIES\n")
print(f"{'':28s} {'shared string':>14s} {'unique string':>14s}")
for wu, lab in [(True, "writes a unit/level"), (False, "does not")]:
    a = int(((dec.writes_unit == wu) & dec.shared).sum())
    b = int(((dec.writes_unit == wu) & ~dec.shared).sum())
    print(f"{lab:28s} {a:14d} {b:14d}")

can_fix  = int((dec.writes_unit & dec.shared).sum())
would_hurt = int((~dec.writes_unit & dec.shared).sum())
print(f"\n  RENDER change could rescue : {can_fix:3d} ({can_fix/n:.1%})")
print(f"  RENDER change would burden : {would_hurt:3d} ({would_hurt/n:.1%})")
print(f"  net if every rescue works and every burden fails: {can_fix-would_hurt:+d} queries")
print(f"\n  median group size where shared: "
      f"{dec.loc[dec.shared,'group_size'].median():.0f}, "
      f"max {dec.group_size.max():.0f}")
dec.to_csv(OUT / "render_decision.tsv", sep="\t", index=False)

OF 789 MATCHABLE QUERIES

                              shared string  unique string
writes a unit/level                     126             29
does not                                180            454

  RENDER change could rescue : 126 (16.0%)
  RENDER change would burden : 180 (22.8%)
  net if every rescue works and every burden fails: -54 queries

  median group size where shared: 20, max 493


In [ ]:
# Cell 16: end-to-end results by subset (the shape of the true record: single
# number or part of a range), all on the string rule. The subsets are read from
# stage1_recall_by_subset.tsv, written by 3_train_retriever. Headroom is the
# share of the gap between the retriever's top-1 and its recall@K that the
# re-ranker closed.

SUB = S1_OUT / "stage1_recall_by_subset.tsv"
assert SUB.exists(), f"{SUB} not found; run 3_train_retriever.ipynb cell 22 first"
sb   = pd.read_csv(SUB, sep="\t", dtype=str)
a2s  = dict(zip(sb.address, sb.subset))
K_REPORT = int(best.K)   # the K cell 15 chose

recs = []
for i, u in enumerate(m_uids):
    recs.append(dict(subset=a2s.get(uid2addr[u], "unknown"), address=uid2addr[u],
                     s1_str_rank=int(s1_str_rank[u]),
                     s1_hit=bool(s1_hit[i]), s2_hit=bool(s2_hit[i])))
bysub = pd.DataFrame(recs)

print(f"END-TO-END BY SUBSET (raw candidates, K={K_REPORT}, STRING rule throughout)\n")
print(f"{'subset':16s} {'n':>4s} {'S1 top-1':>9s} {'S1 rec@K':>9s} {'S1+S2':>8s} "
      f"{'gain':>7s} {'headroom':>9s}")
for name in ["single", "range_first", "range_last", "range_interior",
             "range_written", "ALL"]:
    g = bysub if name == "ALL" else bysub[bysub.subset == name]
    if not len(g): continue
    s1t = float(g.s1_hit.mean())
    s1r = float(((g.s1_str_rank > 0) & (g.s1_str_rank <= K_REPORT)).mean())
    e   = float(g.s2_hit.mean())
    head = (e - s1t) / (s1r - s1t) if s1r > s1t + 1e-9 else float("nan")
    print(f"{name:16s} {len(g):4d} {s1t:9.3f} {s1r:9.3f} {e:8.3f} "
          f"{e-s1t:+7.3f} {head:8.1%}")
print("\nheadroom = share of the gap between Stage 1's own top-1 and its recall@K")
print("that the re-ranker converted. It is now bounded at 100% by construction.")
bysub.to_csv(OUT / "stage2_by_subset.tsv", sep="\t", index=False)

END-TO-END BY SUBSET (raw candidates, K=10, STRING rule throughout)

subset              n  S1 top-1  S1 rec@K    S1+S2    gain  headroom
single            634     0.850     0.927    0.831  -0.019   -24.5%
range_first        33     0.424     0.818    0.818  +0.394   100.0%
range_last         54     0.759     0.944    0.926  +0.167    90.0%
range_interior     20     0.250     0.300    0.250  +0.000     0.0%
range_written      45     0.867     0.911    0.911  +0.044   100.0%
ALL               789     0.812     0.907    0.828  +0.015    16.0%

headroom = share of the gap between Stage 1's own top-1 and its recall@K
that the re-ranker converted. It is now bounded at 100% by construction.


In [ ]:
# Cell 15c: McNemar's exact test per subset. The overall change can hide a
# loss on one subset and a gain on another, so each subset is tested
# separately. Small subsets cannot reach significance, so counts are printed
# beside the p values. Needs cell 16 (bysub).

import math
def mcnemar_exact(b, c):
    m = b + c
    if m == 0: return 1.0
    k = min(b, c)
    return min(1.0, 2 * sum(math.comb(m, i) for i in range(k + 1)) / (2 ** m))

print(f"{'subset':16s} {'n':>4s} {'lost':>5s} {'rescued':>8s} {'net':>5s} "
      f"{'p (exact)':>10s}  verdict")
for name in ["single", "range_first", "range_last", "range_interior",
             "range_written", "ALL"]:
    m = np.ones(len(bysub), bool) if name == "ALL" else (bysub.subset == name).to_numpy()
    if not m.sum(): continue
    a = bysub.s1_hit.to_numpy()[m]; b_ = bysub.s2_hit.to_numpy()[m]
    lost = int((a & ~b_).sum()); resc = int((~a & b_).sum())
    p = mcnemar_exact(lost, resc)
    v = "significant" if p < 0.05 else ("underpowered" if lost + resc < 10 else "no effect")
    print(f"{name:16s} {int(m.sum()):4d} {lost:5d} {resc:8d} {resc-lost:+5d} "
          f"{p:10.4f}  {v}")
print("\nlost    = Stage 1 had it right, Stage 2 moved something else to the top")
print("rescued = Stage 1 had it below rank 1, Stage 2 promoted it")
print("A subset with fewer than 10 disagreements cannot reach p<0.05 even if")
print("every disagreement points the same way, so 'underpowered' is not 'no effect'.")

subset              n  lost  rescued   net  p (exact)  verdict
single            634    31       19   -12     0.1189  no effect
range_first        33     0       13   +13     0.0002  significant
range_last         54     0        9    +9     0.0039  significant
range_interior     20     0        0    +0     1.0000  underpowered
range_written      45     0        2    +2     0.5000  underpowered
ALL               789    31       43   +12     0.2007  no effect

lost    = Stage 1 had it right, Stage 2 moved something else to the top
rescued = Stage 1 had it below rank 1, Stage 2 promoted it
A subset with fewer than 10 disagreements cannot reach p<0.05 even if
every disagreement points the same way, so 'underpowered' is not 'no effect'.


In [ ]:
# Cell 17: the accept/reject threshold and the confusion matrix. tau is the
# score below which the address is deemed unmatchable instead of accepted.
# tau is the max-F1 threshold on the synthetic validation split only, so no
# test-set row is used to choose it. Each query is judged by its top score.
# AUROC does not depend on tau.

vlog = ce.predict(list(zip(val_df["query"], val_df.candidate)),
                  batch_size=PREDICT_BATCH, show_progress_bar=False)
vs = 1.0 / (1.0 + np.exp(-np.asarray(vlog, dtype=np.float64)))
yv = val_df.label.to_numpy()

def f1_at(t):
    p = (vs >= t).astype(int)
    tp = int(((p == 1) & (yv == 1)).sum())
    fp = int(((p == 1) & (yv == 0)).sum()); fn = int(((p == 0) & (yv == 1)).sum())
    return 2 * tp / max(2 * tp + fp + fn, 1)
grid = np.arange(0.05, 1.00, 0.01)
TAU = float(grid[int(np.argmax([f1_at(t) for t in grid]))])
print(f"tau = {TAU:.2f}  (max-F1 on {len(val_df):,} SYNTHETIC validation pairs)")

K_REPORT = int(best.K)
sub = cand_raw[cand_raw["rank"] <= K_REPORT]
top = (sub.sort_values(["uid", "s_model", "rank"],
                       ascending=[True, False, True], kind="mergesort")
          .groupby("uid").head(1).reset_index(drop=True))
top = top.merge(allq[["uid", "address", "population", "kind", "reason"]], on="uid")
# Excluded rows are retrieved, so uid numbering matches allq, but they are
# dropped here before coverage, precision and specificity.
_n_top_excl = int((top.population == "excluded").sum())
print(f"{_n_top_excl} excluded rows dropped before the confusion matrix "
      f"({int((allq.population == 'excluded').sum())} in allq)")
top = top[top.population != "excluded"].reset_index(drop=True)
top["is_truth"] = [r.cand == truth_str.get(r.uid, "\x00") for r in top.itertuples()]

def auroc(pos, neg):
    x = np.concatenate([pos, neg]); r = pd.Series(x).rank().to_numpy()
    n1, n0 = len(pos), len(neg)
    return (r[:n1].sum() - n1*(n1+1)/2) / max(n1*n0, 1)

mp = top.loc[top.population == "matchable",   "s_model"].to_numpy()
up = top.loc[top.population == "unmatchable", "s_model"].to_numpy()
print(f"\nAUROC matchable vs unmatchable (per-query top score): {auroc(mp, up):.4f}")
for k in ["no_number", "no_record"]:
    s = top.loc[top.kind == k, "s_model"].to_numpy()
    if len(s): print(f"  vs {k:10s} (n={len(s):3d}): {auroc(mp, s):.4f}")

def report(t, tau):
    a, m = t.s_model >= tau, t.population == "matchable"
    ca = int((a & t.is_truth & m).sum()); wa = int((a & ~t.is_truth).sum())
    rm = int((~a & m).sum());             cr = int((~a & ~m).sum())
    prec = ca / max(ca + wa, 1); rec = ca / max(int(m.sum()), 1)
    spec = cr / max(int((~m).sum()), 1)
    return dict(tau=round(tau, 2), coverage=round(a.mean(), 4),
                precision=round(prec, 4), recall=round(rec, 4),
                specificity=round(spec, 4),
                f1=round(2*prec*rec/max(prec+rec, 1e-9), 4),
                correct_accept=ca, wrong_accept=wa,
                rejected_matchable=rm, correct_reject=cr)

print(f"\nAT THE FITTED THRESHOLD (tau = {TAU:.2f}, K = {K_REPORT})")
for k, v in report(top, TAU).items():
    print(f"  {k:20s} {v}")
print("\nDIAGNOSTIC SWEEP (not the selection procedure)")
print(pd.DataFrame([report(top, t) for t in
                    [0.1, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95]]).to_string(index=False))

print("\nUNMATCHABLE QUERIES THE MODEL SCORED HIGHEST (adjudicate by hand):")
print(top[top.population == "unmatchable"].nlargest(15, "s_model")
        [["address", "cand", "s_model", "kind"]].to_string(index=False))
top.to_csv(OUT / "stage2_predictions.tsv", sep="\t", index=False)

tau = 0.83  (max-F1 on 6,350 SYNTHETIC validation pairs)
0 excluded rows dropped before the confusion matrix (0 in allq)

AUROC matchable vs unmatchable (per-query top score): 0.9144
  vs no_number  (n=129): 0.9189
  vs no_record  (n= 32): 0.8960

AT THE FITTED THRESHOLD (tau = 0.83, K = 10)
  tau                  0.83
  coverage             0.6779
  precision            0.8975
  recall               0.7326
  specificity          0.9317
  f1                   0.8067
  correct_accept       578
  wrong_accept         66
  rejected_matchable   156
  correct_reject       150

DIAGNOSTIC SWEEP (not the selection procedure)
 tau  coverage  precision  recall  specificity     f1  correct_accept  wrong_accept  rejected_matchable  correct_reject
0.10    0.7779     0.8336  0.7807       0.7205 0.8063             616           123                  95             116
0.30    0.7337     0.8680  0.7668       0.8634 0.8143             605            92                 114             139
0.50    0.7137

In [ ]:
# Cell 17b: write a review sheet for the queries labelled unmatchable, to
# estimate how many of those labels are wrong. The sheet holds every
# unmatchable query the model accepted, plus 20 random rejected ones as a
# control. A person fills in the verdict column. The labels are not corrected
# automatically, because that would tune the test set to agree with the model.

un = top[top.population == "unmatchable"].copy()
acc = un[un.s_model >= TAU]
ctl = un[un.s_model < TAU].sample(min(20, int((un.s_model < TAU).sum())),
                                  random_state=SEED)
sheet = pd.concat([acc.assign(group="accepted"), ctl.assign(group="control")])
sheet = sheet[["group", "kind", "address", "cand", "s_model"]].copy()
sheet["verdict"] = ""       # matchable / unmatchable / unsure
sheet = sheet.sort_values(["group", "s_model"], ascending=[True, False])
sheet.to_csv(OUT / "reject_class_review.tsv", sep="\t", index=False)

print(f"wrote {OUT/'reject_class_review.tsv'}")
print(f"  {len(acc)} accepted (these are the 'wrong accepts' the metric counts)")
print(f"  {len(ctl)} control rows sampled from the rejected side")
print("\nFill the verdict column with matchable / unmatchable / unsure, then the")
print("mislabel rate among the accepted group is the correction to apply to")
print("specificity, reported as a range rather than a point if 'unsure' is large.")

wrote output_stage2/reject_class_review.tsv
  11 accepted (these are the 'wrong accepts' the metric counts)
  20 control rows sampled from the rejected side

Fill the verdict column with matchable / unmatchable / unsure, then the
mislabel rate among the accepted group is the correction to apply to
specificity, reported as a range rather than a point if 'unsure' is large.


In [ ]:
# Cell 17c: the accept/reject metrics under a stated range of label noise.
# Only a sample of the rejected unmatchable rows was reviewed, so the share
# mislabelled there is an estimate with a Wilson interval, and the table gives
# optimistic, point and pessimistic rows.
# The confusion matrix is read from cell 17. The review figures below are
# hand-entered from the manual label review.

import math
def wilson(k, n, z=1.96):
    if n == 0: return (0.0, 1.0)
    p = k/n; d = 1+z*z/n; c = p+z*z/(2*n)
    h = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return ((c-h)/d, (c+h)/d)

_rep = report(top, TAU)                       # cell 17's matrix
CA, WA = _rep["correct_accept"], _rep["wrong_accept"]
RM, CR = _rep["rejected_matchable"], _rep["correct_reject"]
N_MATCH   = int((top.population == "matchable").sum())
N_UNMATCH = int((top.population == "unmatchable").sum())
print(f"from cell 17 at tau {TAU:.2f}: CA {CA}, WA {WA}, RM {RM}, CR {CR}; "
      f"{N_MATCH} matchable, {N_UNMATCH} unmatchable\n")

# Hand-entered review verdicts
N_ACC_MISLABELLED  = 0                   # accepted unmatchable rows judged mislabelled
                                         # in the manual label review
K_CTL, N_CTL_SAMP  = 5, 20               # control sample: 5 of 20 rejected rows were matchable
N_CTL_TOTAL        = CR - 15             # rejected rows never reviewed: the other 15 control
                                         # rows were reviewed and kept, and the 5 found
                                         # matchable are now matchable rows

lo, hi = wilson(K_CTL, N_CTL_SAMP)
print(f"control sample: {K_CTL}/{N_CTL_SAMP} rejected rows were actually matchable")
print(f"  = {K_CTL/N_CTL_SAMP:.0%}, Wilson 95% CI [{lo:.2f}, {hi:.2f}]")
print(f"  -> {N_CTL_TOTAL*lo:.0f} to {N_CTL_TOTAL*hi:.0f} mislabels among the "
      f"{N_CTL_TOTAL} rejected\n")

print(f"{'scenario':14s} {'mislabels':>10s} {'precision':>10s} {'recall':>8s} "
      f"{'F1':>7s} {'specificity':>12s}")
for lab, f in [("as reported", None), ("optimistic", lo),
               ("point est", K_CTL/N_CTL_SAMP), ("pessimistic", hi)]:
    if f is None:
        p, r = CA/(CA+WA), CA/N_MATCH
        sp, mis = CR/N_UNMATCH, 0
    else:
        mis_rej = N_CTL_TOTAL*f
        mis = N_ACC_MISLABELLED + mis_rej
        ca, wa = CA+N_ACC_MISLABELLED, WA-N_ACC_MISLABELLED
        true_match = N_MATCH + mis
        true_unmatch = N_UNMATCH - mis
        p = ca/max(ca+wa, 1); r = ca/true_match
        sp = (CR-mis_rej)/max(true_unmatch, 1e-9)
    f1 = 2*p*r/max(p+r, 1e-9)
    print(f"{lab:14s} {mis:10.0f} {p:10.4f} {r:8.4f} {f1:7.4f} {sp:12.4f}")

print("\nspecificity of exactly 1.000 is arithmetic, not evidence: it follows from")
print("assuming ALL accepted rows were mislabelled. Record a verdict per accepted")
print("row and re-run before quoting it.")

from cell 17 at tau 0.83: CA 578, WA 66, RM 156, CR 150; 789 matchable, 161 unmatchable

control sample: 5/20 rejected rows were actually matchable
  = 25%, Wilson 95% CI [0.11, 0.47]
  -> 15 to 63 mislabels among the 135 rejected

scenario        mislabels  precision   recall      F1  specificity
as reported             0     0.8975   0.7326  0.8067       0.9317
optimistic             15     0.8975   0.7188  0.7983       0.9246
point est              34     0.8975   0.7025  0.7881       0.9136
pessimistic            63     0.8975   0.6782  0.7726       0.8874

specificity of exactly 1.000 is arithmetic, not evidence: it follows from
assuming ALL accepted rows were mislabelled. Record a verdict per accepted
row and re-run before quoting it.


In [ ]:
# Cell 18: write a manifest of the model, index and test set (with its sha1)
# that produced the numbers above, to OUT/stage2_run_manifest.json.

manifest = dict(
    timestamp=time.strftime("%Y-%m-%d %H:%M:%S"),
    stage1=dict(model=str(S1_MODEL), mtime=S1_MTIME, index=str(EMB_NPY),
                top1=round(s1_top1, 4)),
    stage2=dict(base=CE_MODEL, epochs=EPOCHS, batch=BATCH, lr=LR,
                train_pairs=len(train_df), val_pairs=len(val_df),
                mine_n=len(mine), mine_k=MINE_K, neg_per_pos=NEG_PER_POS),
    test_set=dict(file=str(MISSPELLINGS_TSV),
                  sha1=hashlib.sha1(Path(MISSPELLINGS_TSV).read_bytes()).hexdigest(),
                  n_rows=int(len(allq)), n_excluded=int(n_excluded)),
    eval=dict(n_matchable=int(len(m_uids)),
              n_unmatchable=int((allq.population == "unmatchable").sum()),
              retrieve_k=RETRIEVE_K, k_max=K_MAX, k_reported=int(best.K),
              tau=TAU, top1_string=round(float(best.top1_str), 4),
              top1_pid=round(float(best.top1_pid), 4)),
)
(OUT / "stage2_run_manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
print(f"\nartefacts in {OUT}:")
for p in sorted(OUT.glob("*.tsv")) + sorted(OUT.glob("*.json")):
    print(f"  {p.name:34s} {p.stat().st_size/1024:8.1f} KB")

{
  "timestamp": "2026-09-08 02:04:01",
  "stage1": {
    "model": "output_stage1/model",
    "mtime": 1788792286.0973577,
    "index": "output_stage1/emb_store.npy",
    "top1": 0.5463
  },
  "stage2": {
    "base": "cross-encoder/ms-marco-MiniLM-L6-v2",
    "epochs": 2,
    "batch": 16,
    "lr": 2e-05,
    "train_pairs": 54679,
    "val_pairs": 6350,
    "mine_n": 8000,
    "mine_k": 32,
    "neg_per_pos": 8
  },
  "test_set": {
    "file": "output/misspellings.tsv",
    "sha1": "d48fdfe227fd6ef7828384f020e3042d18873101",
    "n_rows": 950,
    "n_excluded": 0
  },
  "eval": {
    "n_matchable": 789,
    "n_unmatchable": 161,
    "retrieve_k": 500,
    "k_max": 100,
    "k_reported": 10,
    "tau": 0.8300000000000002,
    "top1_string": 0.8276,
    "top1_pid": 0.5323
  }
}

artefacts in output_stage2:
  end_to_end_per_query.tsv               14.9 KB
  label_audit_sample.tsv                  6.9 KB
  label_audit_sample_v2.tsv               8.3 KB
  misspellings_diff.tsv              